# Budget Forecasting Notebook

This notebook demonstrates budget forecasting using linear regression on historical spending data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from datetime import datetime, timedelta

# Load and prepare time series data
transactions = pd.read_csv('../datasets/raw/transactions.csv')
monthly = transactions.groupby('date')['amount'].sum().reset_index()
monthly['date'] = pd.to_datetime(monthly['date'])
monthly = monthly.sort_values('date').reset_index(drop=True)
monthly['time_index'] = np.arange(len(monthly))

print('Monthly data:')
print(monthly)

In [ ]:
# Train linear regression model
X = monthly[['time_index']].values
y = monthly['amount'].values
model = LinearRegression()
model.fit(X, y)

predictions = model.predict(X)
print(f"R-squared: {model.score(X, y):.4f}")
print(f"Coefficient: {model.coef_[0]:.2f}")
print(f"Intercept: {model.intercept_:.2f}")

# Plot historical vs fitted
plt.figure(figsize=(10, 5))
plt.plot(monthly['date'], y, marker='o', label='Actual')
plt.plot(monthly['date'], predictions, marker='x', linestyle='--', label='Fitted')
plt.title('Historical Spending vs Fitted Values')
plt.xlabel('Date')
plt.ylabel('Amount (INR)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Forecast next 6 months
horizon = 6
last_index = monthly['time_index'].iloc[-1]
last_date = monthly['date'].iloc[-1]
future_indices = np.arange(last_index + 1, last_index + 1 + horizon).reshape(-1, 1)
future_predictions = model.predict(future_indices)

future_dates = [last_date + timedelta(days=30 * i) for i in range(1, horizon + 1)]

forecast_df = pd.DataFrame({
    'date': future_dates,
    'predicted': future_predictions.round(2),
    'lower_bound': (future_predictions * 0.9).round(2),
    'upper_bound': (future_predictions * 1.1).round(2)
})

print('Forecast:')
print(forecast_df)